[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsonmen/bias-and-variance/blob/main/computer-vision/ImageClassification/CatsvsDogs/VGG16TransferLearning.ipynb)

# Abstract

- Goal: Fine-Tune VGG16 on Cats vs Dogs dataset and reach a good accuracy score

- Dataset: [Cats and Dogs Classification Dataset (Kaggle)](https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset)

- Project Details:

    I'm using pre-trained VGG16 on IMAGENET1K dataset and fine-tune it for binary classification tasks on Cats vs Dogs dataset. Also, I made a form for predicting your own image or select an image from the dataset.

  
- Best result: 0.9722 (Accuracy Score of VGG16)

- Sections:
    - [Imports](#Imports)
    - [Model Parameters](#Model-Parameters)
    - [Utils](#Utils)
    - [Dataset](#Dataset)
    - [Modeling](#Modeling)
    - [Prediction](#Prediction)
        - [Load Model](#Load-Model)
        - [Setup Form](#Setup-Form)
        - [Prediction Form](#Prediction-Form)

## Problems & Solutions
- Problem 1: Model is underfitted or model has high bias

    **Symptoms:**

    - Model's train/validation loss and accuracy are low

    **Root Cause:**

    - Big size of model classifier

    **Solution:**

    - Reduce size of model classifier. Instead of 3 fc -> 2 fc, hidden dim between 2 fc 4096 -> 512

# Download & Install Dependencies

In [ ]:
# Install modules (if needed)
!pip install torch torchvision matplotlib pillow numpy tqdm ipywidgets ipython

In [ ]:
# Dataset Downloading
!mkdir data
!curl -L -o ./data/dataset.zip https://www.kaggle.com/api/v1/datasets/download/bhavikjikadara/dog-and-cat-classification-dataset
!unzip -qq ./data/dataset.zip -d ./data
!rm ./data/dataset.zip
!mv ./data/PetImages ./data/catsvsdogs

In [2]:
# Model Downloading
!mkdir models
from huggingface_hub import snapshot_download

PROJECT_NAME = "ImageClassificationCatsVSDogs"
MODEL_FOLDER = "vgg16"
repo_id = f"jsonmen/{PROJECT_NAME}"

snapshot_download(
    repo_id=repo_id,
    local_dir="./models",
    allow_patterns=[f"{MODEL_FOLDER}/*"],
    token=False  # No token needed for public repos
)
print(f"Downloaded models folder from {repo_id} to ./models")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

vgg16_model.pt:   0%|          | 0.00/110M [00:00<?, ?B/s]

Downloaded models folder from jsonmen/ImageClassificaitionCatsVSDogs to ./models


# Imports

In [ ]:
import torch
from torch import nn, Tensor
import torchvision
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision.transforms import v2
from PIL import Image
import numpy as np
from tqdm import tqdm
import sys
import warnings
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# Model Parameters

In [ ]:
BATCH_SIZE = 32
LR = 3e-4
EPOCH = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

# Utils

In [ ]:
def binary_accuracy(y_pred, y_true):
    """
    Compute accuracy for binary classification.
    """
    # sigmoid → probabilities
    probs = torch.sigmoid(y_pred)
    preds = torch.round(probs)
    correct = (preds == y_true).float()
    acc = correct.sum() / len(correct)
    return acc

def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    """
    Train model for one epoch.
    """
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(dataloader, desc="Train", leave=False)

    for inputs, targets in progress_bar:
        inputs = inputs.to(device)
        targets = targets.to(device).float().unsqueeze(1)  # Shape: (batch, 1)

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = loss_fn(outputs, targets)
        acc = binary_accuracy(outputs, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_acc += acc.item() * inputs.size(0)

        avg_loss = running_loss / ((progress_bar.n + 1) * inputs.size(0))
        avg_acc = running_acc / ((progress_bar.n + 1) * inputs.size(0))
        progress_bar.set_postfix(loss=avg_loss, acc=avg_acc)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, loss_fn, device):
    """
    Evaluate model.
    """
    model.eval()
    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(dataloader, desc="Val", leave=False)

    with torch.no_grad():
        for inputs, targets in progress_bar:
            inputs = inputs.to(device)
            targets = targets.to(device).float().unsqueeze(1)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            acc = binary_accuracy(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            running_acc += acc.item() * inputs.size(0)

            avg_loss = running_loss / ((progress_bar.n + 1) * inputs.size(0))
            avg_acc = running_acc / ((progress_bar.n + 1) * inputs.size(0))
            progress_bar.set_postfix(loss=avg_loss, acc=avg_acc)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def train_loop(model, train_loader, val_loader, optimizer, loss_fn, device, epochs):
    """
    Run full training loop.
    """
    # Example usage:
    # train_loop(
    #     model,
    #     train_loader,
    #     val_loader,
    #     optimizer,
    #     loss_fn,
    #     DEVICE,
    #     EPOCH
    # )
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, loss_fn, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, loss_fn, device
        )

        print(
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

In [ ]:
import warnings
warnings.filterwarnings(
    action='ignore',
    category=UserWarning,
)


# Dataset

In [ ]:
class SafeImageFolder(torchvision.datasets.ImageFolder): # This class its ImageFolder but with broken image filtering 
    def __init__(self, root, transform=None, target_transform=None):
        super().__init__(root, transform=transform, target_transform=target_transform)
        self.samples = [
            (path, class_idx) for path, class_idx in self.samples
            if self._is_valid_image(path)
        ]

    def _is_valid_image(self, path):
        try:
            with Image.open(path) as img:
                img.verify()
            return True
        except Exception:
            return False

In [ ]:
image_transforms = v2.Compose([v2.Resize((224, 224)), 
                        v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]) 

In [ ]:
dataset = SafeImageFolder("./data/catsvsdogs", transform=image_transforms, target_transform=lambda x: torch.tensor(x, dtype=torch.float32))
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Modeling

Few words about **VGG‑16**:

**VGG‑16** refers to the Visual Geometry Group network with **16 weight layers** (13 conv + 3 FC).
It stacks **small 3×3 convolutional filters** in increasing depth (64→128→256→512) to learn hierarchical features.

At the top, it has three fully‑connected layers (4096→4096→1000), though for my task i'd replace the final 3 FC to 2 FC and hidden dim between 2 FC 4096 -> 512 (512→1)

**Image that makes sense of VGG‑16’s uniform conv stack:**

<img src="https://cdn.analyticsvidhya.com/wp-content/uploads/2024/09/vgg16-neural-network.webp" alt="VGG16 architecture diagram" width="500"/>

**Image comparing VGG‑16 filter sizes vs. other nets:**

<img src="https://www.researchgate.net/publication/350550608/figure/fig3/AS:1007769725452289@1617282434523/The-Difference-Architecture-between-AlexNet-and-VGG16-Models.png" alt="VGG16 vs AlexNet filter comparison" width="500"/>

In [ ]:
model = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
model.classifier = nn.Sequential(
    nn.Linear(in_features=25088, out_features=512, bias=True),
    nn.ReLU(inplace=True),
    nn.Linear(in_features=512, out_features=1, bias=True),
)
for param in model.parameters():
    param.requires_grad = True
model = model.to(DEVICE)

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [ ]:
train_loop(model, train_dataloader, val_dataloader, optimizer, loss_fn, DEVICE, EPOCH)

In [ ]:
torch.save(model.state_dict(), "./models/vgg16/vgg16_model.pt") # Save model weights

# Prediction

## Load Model

In [ ]:
trained_model = torchvision.models.vgg16(weights=None)
trained_model.classifier = nn.Sequential(
    nn.Linear(in_features=25088, out_features=512, bias=True),
    nn.ReLU(inplace=True),
    nn.Linear(in_features=512, out_features=1, bias=True),
)
trained_model.load_state_dict(torch.load("./models/vgg16/vgg16_model.pt", map_location=torch.device('cpu')))

for param in trained_model.parameters():
    param.requires_grad = True
trained_model = trained_model.cpu()
trained_model.eval()

## Setup Form 

In [ ]:
t1_output = widgets.Output()

image_id = widgets.BoundedIntText(
    value=7,
    min=0,
    max=len(val_dataset),
    step=1,
    description='Image id from dataset:',
    disabled=False,
    style={'description_width': 'initial'}
)
def t1_func(b):
    image_id_i = image_id.value
    cat_or_dog = lambda x: "Cat" if torch.tanh(x).item() <= 0 else "Dog"  
    with t1_output:
        clear_output()
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(val_dataset[image_id_i][0].permute(1, 2, 0).numpy())
        plt.title("Selected image")
        plt.xticks([])
        plt.yticks([])
        plt.subplot(1, 2, 2)
        plt.axis('off')
        prediction = trained_model(val_dataset[image_id_i][0].unsqueeze(0))
        info = f"""Raw Model Prediction: {prediction.item():.3f}

Model Prediction in Words: {cat_or_dog(prediction)}
Model Prediction Confidence: {torch.tanh(prediction).abs().item():.2%}"""

        plt.text(0, 1, info, fontsize=12, va='top')
        plt.tight_layout()
        plt.show()
        

t1_button = widgets.Button(description="Classify")
t1_button.on_click(t1_func)
t1 = widgets.VBox([image_id, t1_button, t1_output])

In [ ]:
t2_output = widgets.Output()

image_file = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    desctiption="Image file Upload: "
)
def t2_func(b):
    filename = list(image_file.value.keys())[0]
    bytes_content = image_file.value[filename]['content']
    pil_image = Image.open(io.BytesIO(bytes_content))
    cat_or_dog = lambda x: "Cat" if torch.tanh(x).item() <= 0 else "Dog" 
    with t2_output:
        clear_output()
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(pil_image)
        plt.title("Selected image")
        plt.xticks([])
        plt.yticks([])
        plt.subplot(1, 2, 2)
        plt.axis('off')
        prediction = trained_model(image_transforms(pil_image).unsqueeze(0))
        info = f"""Raw Model Prediction: {prediction.item():.3f}

Model Prediction in Words: {cat_or_dog(prediction)}
Model Prediction Confidence: {torch.tanh(prediction).abs().item():.2%}"""

        plt.text(0, 1, info, fontsize=12, va='top')
        plt.tight_layout()
        plt.show()
t2_button = widgets.Button(description="Classify")
t2_button.on_click(t2_func)

t2 = widgets.VBox([image_file, t2_button, t2_output])

In [ ]:
form = widgets.Tab()
form.children = [t1, t2]
form.set_title(0, 'Select from dataset')
form.set_title(1, 'Upload image file')

## Prediction Form

In [ ]:
display(form)